<table width="100%" bgcolor="#f4f5f6" style="width: 100%; background-color: #f4f5f6; border-left: 6px solid #ff9900; border-radius: 8px; border-collapse: collapse;">
  <tr>
    <td width="110" align="center" valign="middle" style="padding: 15px;">
      <img src="https://github.com/sansalamanca/shining-path_DP700/raw/main/data/logo%20libro.jpg" width="85" style="border-radius: 8px; display: block;">
    </td>
    <td valign="middle" style="padding: 15px 20px 15px 5px; font-family: Arial, sans-serif;">
      <p style="margin: 0 0 4px 0; font-size: 13px; color: #555555;">Excerpt from the Amazon book</p>
      <h2 style="margin: 4px 0 10px 0; font-size: 20px; font-weight: 900; color: #111111; border: none; padding: 0;">THE SHINING PATH TO DATA ENGINEER CERTIFICATE</h2>
      <p style="margin: 0; font-size: 14px; font-weight: bold;">
        <a href="https://www.amazon.com" target="_blank" style="color: #b12704; text-decoration: underline;">🛒 Buy on Amazon &rarr;</a>
      </p>
    </td>
  </tr>
</table>

In [1]:
# 1. Install PySpark and Delta Lake
!pip install -q --no-warn-conflicts pyspark==3.5.3 delta-spark==3.2.1

# 2. Clone the repository containing Lakehouse/Files/sales.csv
!rm -rf shining-path_DP700
!git clone -q https://github.com/sansalamanca/shining-path_DP700.git

# 3. Initialize SparkSession with Delta Lake support
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip

builder = SparkSession.builder \
    .appName("Bronze_Layer_Ingestion") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")

spark = configure_spark_with_delta_pip(builder).getOrCreate()
print("SparkSession with Delta Lake is ready!")


  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
SparkSession with Delta Lake is ready!


In [2]:
from pyspark.sql.functions import current_timestamp, input_file_name

# 1. Batch Load (Append-only): Read raw data from the Landing Zone (Files/)
# Note: escape='"' ensures product names with commas (e.g., "Mountain-100 Silver, 44") are parsed properly
df_landing = spark.read.format("csv") \
    .option("header", "true") \
    .option("escape", '"') \
    .load("shining-path_DP700/Lakehouse/Files/sales.csv")

# 2. Add audit metadata columns (ingestion_timestamp and source_file)
df_bronze = df_landing \
    .withColumn("ingestion_timestamp", current_timestamp()) \
    .withColumn("source_file", input_file_name())

# 3. Save to the Bronze layer inside Tables/ in Delta format (append-only, no filtering)
bronze_table_path = "shining-path_DP700/Lakehouse/Tables/stg_sales_bronze"

df_bronze.write \
    .format("delta") \
    .mode("append") \
    .option("mergeSchema", "true") \
    .save(bronze_table_path)

print("Raw sales data successfully appended to Bronze table: stg_sales_bronze!")

Raw sales data successfully appended to Bronze table: stg_sales_bronze!


In [3]:
# Read back the Bronze Delta table to verify data and audit columns
df_verify = spark.read.format("delta").load("shining-path_DP700/Lakehouse/Tables/stg_sales_bronze")

df_verify.show(5, truncate=False)
df_verify.printSchema()

+----------------+--------------------+----------+--------------+-----------------------------+-----------------------+--------+---------+---------+-------------------------+------------------------------------------------------------+
|SalesOrderNumber|SalesOrderLineNumber|OrderDate |CustomerName  |EmailAddress                 |Item                   |Quantity|UnitPrice|TaxAmount|ingestion_timestamp      |source_file                                                 |
+----------------+--------------------+----------+--------------+-----------------------------+-----------------------+--------+---------+---------+-------------------------+------------------------------------------------------------+
|SO43701         |1                   |2019-07-01|Christy Zhu   |christy12@adventure-works.com|Mountain-100 Silver, 44|1       |3399.99  |271.9992 |2026-11-16 21:36:22.22217|file:///content/shining-path_DP700/Lakehouse/Files/sales.csv|
|SO43704         |1                   |2019-07-01|Julio 